# Bridge Risk Intelligence Prototype

This notebook explains the bridge risk prototype with short notes and simple code comments.

Project logic:

```text
Bridge inspection data
        ↓
Clean and match yearly records
        ↓
Create deterioration target
        ↓
Train model
        ↓
Score bridge risk
        ↓
Create priority list
        ↓
Engineer review
```

This tool supports early screening only. It does not replace professional bridge inspection.

## 1. Problem Logic

```text
Current bridge condition
        ↓
Predict next year rating change
        ↓
Rank bridges by risk
        ↓
Review high-priority bridges first
```

### Goal, Input, Output

```text
Goal
Predict whether a bridge rating may decrease next year.

Input
Current year bridge inspection data.

Output
Risk score and priority ranking.
```

## 2. Setup

```text
Find project root
        ↓
Add project root to Python path
        ↓
Import project modules
```

In [ ]:
# Load Path to build file paths safely.
from pathlib import Path

# Load sys so Python can find local project modules.
import sys

# Load pandas to read output tables.
import pandas as pd

# Load Plotly Express to create quick visual charts.
import plotly.express as px

# Use the parent folder if this notebook is running inside notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Add the project root so imports like src.bridge_risk_pipeline work.
sys.path.insert(0, str(PROJECT_ROOT))

# Show the project root to confirm the notebook is using the right folder.
print(PROJECT_ROOT)

## 3. Run Pipeline

```text
Raw data
        ↓
Pipeline
        ↓
Outputs, reports, model files
```

### Goal, Input, Output

```text
Goal
Run the full ML workflow once.

Input
Raw bridge files from data/.

Output
Priority list, metrics reports, and trained model.
```

In [ ]:
# Load the main pipeline function from the source code.
from src.bridge_risk_pipeline import run_pipeline

# Run fast mode so the demo finishes quickly.
results = run_pipeline(PROJECT_ROOT, fast=True)

## 4. Read Priority List

```text
Priority CSV
        ↓
DataFrame
        ↓
Top risky bridges
```

### Goal, Input, Output

```text
Goal
Check the bridges ranked highest by the model.

Input
outputs/california_bridge_2026_priority_list.csv

Output
Top 10 priority bridges.
```

In [ ]:
# Build the path to the priority list file.
priority_path = PROJECT_ROOT / 'outputs' / 'california_bridge_2026_priority_list.csv'

# Read the ranked bridge list into a DataFrame.
priority = pd.read_csv(priority_path)

# Display the top 10 bridges with the highest predicted risk.
priority.head(10)

## 5. Check Target Rate

```text
Target summary
        ↓
Deterioration rate by year transition
        ↓
Class imbalance check
```

### Goal, Input, Output

```text
Goal
Check how rare deterioration is in each year transition.

Input
reports/target_summary.csv

Output
Bar chart of deterioration rate.
```

In [ ]:
# Read the target summary report.
target_summary = pd.read_csv(PROJECT_ROOT / 'reports' / 'target_summary.csv')

# Create a bar chart to compare deterioration rate by transition.
fig = px.bar(
    target_summary,
    x='transition',
    y='deterioration_rate',
    title='Observed deterioration rate by transition'
)

# Show the chart in the notebook.
fig.show()

## 6. Map Top Priority Bridges

```text
Priority list
        ↓
Keep rows with location
        ↓
Map top 300 bridges
```

### Goal, Input, Output

```text
Goal
Show where high-priority bridges are located.

Input
Top 300 bridges with latitude and longitude.

Output
Interactive map with risk color.
```

In [ ]:
# Keep top bridge records that have map coordinates.
map_data = priority.dropna(subset=['latitude', 'longitude']).head(300)

# Create a map where color shows predicted deterioration probability.
fig = px.scatter_map(
    map_data,
    lat='latitude',
    lon='longitude',
    color='predicted_deterioration_probability',
    hover_name='bridge_id',
    hover_data=[
        'FACILITY_CARRIED_007',
        'LOCATION_009',
        'LOWEST_RATING',
        'BRIDGE_CONDITION',
        'priority_rank'
    ],
    zoom=4.5,
    height=600,
    title='Top 300 bridge priorities'
)

# Show the map in the notebook.
fig.show()

## 7. Presentation Summary

```text
Bridge data
        ↓
ML risk ranking
        ↓
Priority list
        ↓
Engineer review
```

Main point: the model does not make the final safety decision.  
It only helps engineers find records that may need earlier review.